# Notebook 09 - Real Auckland GTFS-Realtime Validation

This notebook validates the real Auckland GTFS-Realtime and Open-Meteo integration layer for the MSE907 capstone project. It uses the canonical Parquet outputs from the Data Storage module when available, while preserving raw daily CSV loading as an audit fallback.

The default validation dataset is the complete-day model baseline. The all-file cleaned dataset remains available for descriptive and dashboard coverage. Notebook 09 does not train machine-learning models, redesign the decision engine, implement SUMO, or build the Streamlit dashboard.


## Validation Flow

1. Load repaired GTFS-Realtime daily files when available, with a fallback to repaired log files.
2. Clean headers, duplicate header rows, timestamps, booleans, and numeric delay fields.
3. Filter unreasonable delay outliers.
4. Create time features and delay-risk categories.
5. Merge route metadata from GTFS Static and validate route match rate.
6. Fetch hourly Open-Meteo weather and align it to GTFS UTC timestamps.
7. Generate operational recommendations and dashboard-ready tables in memory.
8. Regenerate dashboard-ready CSV outputs only from the documented input files and record the resulting Git changes.


In [1]:
# Core imports and project initialization

from pathlib import Path

import numpy as np
import pandas as pd
import json
from urllib.parse import urlencode
from urllib.request import urlopen

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_PARQUET = DATA_PROCESSED / "parquet"
DATA_DUCKDB = DATA_PROCESSED / "duckdb"
GTFS_REALTIME_DIR = DATA_RAW / "gtfs_realtime"
GTFS_DAILY_DIR = GTFS_REALTIME_DIR / "daily"
GTFS_STATIC_DIR = DATA_RAW / "gtfs_static"
SUMMARY_DIR = DATA_PROCESSED / "summaries"

ALL_FILE_PARQUET_PATH = DATA_PARQUET / "gtfs_realtime_cleaned.parquet"
MODEL_BASELINE_PARQUET_PATH = DATA_PARQUET / "gtfs_realtime_model_baseline.parquet"
DUCKDB_PATH = DATA_DUCKDB / "gtfs_realtime.duckdb"
COMPLETENESS_SUMMARY_PATH = SUMMARY_DIR / "gtfs_realtime_collection_completeness.csv"

# Legacy paths are kept only for audit awareness. The canonical paths above are preferred.
LEGACY_CLEANED_PARQUET_PATH = DATA_PROCESSED / "gtfs_realtime_cleaned.parquet"
LEGACY_MODEL_BASELINE_PARQUET_PATH = DATA_PROCESSED / "gtfs_realtime_model_baseline.parquet"

print("Project root:", PROJECT_ROOT)
print("GTFS-Realtime folder exists:", GTFS_REALTIME_DIR.exists())
print("Daily GTFS-Realtime folder exists:", GTFS_DAILY_DIR.exists())
print("GTFS Static folder exists:", GTFS_STATIC_DIR.exists())
print("Canonical Parquet folder exists:", DATA_PARQUET.exists())
print("DuckDB path exists:", DUCKDB_PATH.exists())


Project root: D:\Yoobee\Term 3\Capstone\ai-dss-auckland-public-transport
GTFS-Realtime folder exists: True
Daily GTFS-Realtime folder exists: True
GTFS Static folder exists: True
Canonical Parquet folder exists: True
DuckDB path exists: True


## Step 1 - Locate Real Auckland GTFS-Realtime Inputs

The preferred source is the repaired daily GTFS-Realtime folder because it represents the ongoing real Auckland feed collection. The validation log remains available as a fallback.


In [2]:
# Locate candidate input files without changing them

validation_log_path = GTFS_REALTIME_DIR / "gtfs_realtime_log_validation.csv"
repaired_log_path = GTFS_REALTIME_DIR / "gtfs_realtime_log.csv"
daily_files = sorted(GTFS_DAILY_DIR.glob("gtfs_realtime_*.csv")) if GTFS_DAILY_DIR.exists() else []

candidate_parquet_paths = {
    "all_file": ALL_FILE_PARQUET_PATH,
    "model_baseline": MODEL_BASELINE_PARQUET_PATH,
}

for label, path in candidate_parquet_paths.items():
    print(f"Canonical {label} Parquet exists:", path.exists())
    if path.exists():
        print(f"Canonical {label} Parquet size MB:", round(path.stat().st_size / 1024 / 1024, 2))

print("Completeness summary exists:", COMPLETENESS_SUMMARY_PATH.exists())
print("DuckDB database exists:", DUCKDB_PATH.exists())
print("Legacy cleaned Parquet exists:", LEGACY_CLEANED_PARQUET_PATH.exists())
print("Legacy model-baseline Parquet exists:", LEGACY_MODEL_BASELINE_PARQUET_PATH.exists())

print("Daily files found:", len(daily_files))
if daily_files:
    print("First daily file:", daily_files[0].name)
    print("Last daily file:", daily_files[-1].name)

print("Validation log exists:", validation_log_path.exists())
print("Repaired full log exists:", repaired_log_path.exists())


Canonical all_file Parquet exists: True
Canonical all_file Parquet size MB: 51.08
Canonical model_baseline Parquet exists: True
Canonical model_baseline Parquet size MB: 44.36
Completeness summary exists: True
DuckDB database exists: True
Legacy cleaned Parquet exists: True
Legacy model-baseline Parquet exists: True
Daily files found: 31
First daily file: gtfs_realtime_2026-05-06.csv
Last daily file: gtfs_realtime_2026-06-05.csv
Validation log exists: True
Repaired full log exists: True


In [21]:
# Input selection
# Use "model_baseline" for quality-controlled validation/evaluation.
# Use "all_file" for descriptive/dashboard coverage across all retained collection files.
# Use "auto" to prefer model-baseline Parquet, then all-file Parquet, then daily CSV fallback.

INPUT_DATASET = "all_file"  # options: "auto", "model_baseline", "all_file", "daily", "validation_log", "repaired_log"
MAX_DAILY_FILES = None  # set to a small integer for a quick trial run; None uses all daily files

selected_parquet_path = None
selected_gtfs_files = []
requested_input_source = None

if INPUT_DATASET == "auto":
    selected_parquet_path = MODEL_BASELINE_PARQUET_PATH if MODEL_BASELINE_PARQUET_PATH.exists() else ALL_FILE_PARQUET_PATH
    selected_gtfs_files = daily_files[:MAX_DAILY_FILES] if MAX_DAILY_FILES else daily_files
    requested_input_source = "model_baseline_parquet_else_all_file_parquet_else_daily_csv"
elif INPUT_DATASET == "model_baseline":
    selected_parquet_path = MODEL_BASELINE_PARQUET_PATH
    requested_input_source = "model_baseline_parquet"
elif INPUT_DATASET == "all_file":
    selected_parquet_path = ALL_FILE_PARQUET_PATH
    requested_input_source = "all_file_parquet"
elif INPUT_DATASET == "daily":
    selected_gtfs_files = daily_files[:MAX_DAILY_FILES] if MAX_DAILY_FILES else daily_files
    requested_input_source = "daily_csv"
elif INPUT_DATASET == "validation_log":
    selected_gtfs_files = [validation_log_path]
    requested_input_source = "validation_log_csv"
elif INPUT_DATASET == "repaired_log":
    selected_gtfs_files = [repaired_log_path]
    requested_input_source = "repaired_log_csv"
else:
    raise ValueError("INPUT_DATASET must be 'auto', 'model_baseline', 'all_file', 'daily', 'validation_log', or 'repaired_log'.")

if selected_parquet_path is None and not selected_gtfs_files:
    raise FileNotFoundError("No GTFS-Realtime input files were selected. Check canonical Parquet paths or data/raw/gtfs_realtime.")

print("Selected input dataset:", INPUT_DATASET)
print("Requested input source:", requested_input_source)
print("Selected Parquet path:", selected_parquet_path)
print("Selected Parquet exists:", selected_parquet_path.exists() if selected_parquet_path else None)
print("Selected CSV fallback files:", len(selected_gtfs_files))
print("CSV fallback preview:", [path.name for path in selected_gtfs_files[:5]])


Selected input dataset: all_file
Requested input source: all_file_parquet
Selected Parquet path: D:\Yoobee\Term 3\Capstone\ai-dss-auckland-public-transport\data\processed\parquet\gtfs_realtime_cleaned.parquet
Selected Parquet exists: True
Selected CSV fallback files: 0
CSV fallback preview: []


### Sanity Check - Input Discovery

Expected result: the cleaned Parquet file is detected when available, and daily GTFS-Realtime CSV files are also listed as fallback evidence. If Parquet cannot be read in the current environment, the notebook should clearly report the fallback reason and continue with daily CSV files.


## Step 2 - Load and Clean GTFS-Realtime Records

This step standardizes column names, removes duplicate header rows, converts delay values to numbers, converts timestamps to UTC datetimes, and keeps only records with usable delay and collection time values.


In [22]:
# Safe GTFS-Realtime loader

REQUIRED_GTFS_COLUMNS = [
    "collection_time_utc", "entity_id", "trip_id", "route_id", "direction_id",
    "start_time", "start_date", "timestamp", "delay_seconds", "is_deleted", "delay_minutes",
]

CANONICAL_PARQUET_COLUMNS = [
    "collection_date", "source_file", "collection_day_status", "collection_coverage_hours",
    "is_partial_day", "is_model_baseline_day", "agency_id", "route_short_name", "route_long_name",
    "route_type", "service_type", "route_display_name", "is_special_route",
]

FINAL_STORAGE_COLUMNS = [
    "collection_day_status", "collection_coverage_hours", "is_partial_day", "is_model_baseline_day",
    "service_type", "route_display_name", "is_special_route",
]

def clean_column_names(frame):
    frame = frame.copy()
    frame.columns = frame.columns.astype(str).str.strip().str.lower().str.replace(" ", "_", regex=False)
    return frame

def remove_duplicate_header_rows(frame):
    frame = frame.copy()
    for column in frame.columns:
        frame = frame[frame[column].astype(str).str.strip().str.lower() != column]
    return frame

def parse_utc_datetime(series):
    """Parse UTC timestamps consistently across pandas versions used in project environments."""
    try:
        parsed = pd.to_datetime(series, errors="coerce", utc=True, format="mixed")
    except TypeError:
        parsed = pd.to_datetime(series, errors="coerce", utc=True)

    if parsed.notna().sum() == 0 and series.notna().sum() > 0:
        parsed = pd.to_datetime(series, errors="coerce", utc=True)

    return parsed

def validate_required_columns(frame, required_columns, label):
    missing_columns = sorted(set(required_columns) - set(frame.columns))
    if missing_columns:
        raise ValueError(f"Missing required {label} columns: {missing_columns}")

def normalize_realtime_types(frame):
    frame = frame.copy()

    for column in ["direction_id", "timestamp", "delay_seconds", "delay_minutes", "collection_coverage_hours"]:
        if column in frame.columns:
            frame[column] = pd.to_numeric(frame[column], errors="coerce")

    frame["collection_time_utc"] = parse_utc_datetime(frame["collection_time_utc"])

    for column in ["is_deleted", "is_partial_day", "is_model_baseline_day", "is_special_route"]:
        if column in frame.columns and frame[column].dtype != bool:
            normalized = frame[column].astype(str).str.strip().str.upper()
            frame[column] = normalized.map({"TRUE": True, "FALSE": False, "1": True, "0": False})

    if "collection_date" not in frame.columns:
        frame["collection_date"] = frame["collection_time_utc"].dt.date.astype(str)

    return frame

def load_single_gtfs_file(path):
    frame = pd.read_csv(path, dtype=str, low_memory=False)
    frame = clean_column_names(frame)
    frame = remove_duplicate_header_rows(frame)
    frame["source_file"] = path.name
    return frame

def load_gtfs_realtime_files(paths):
    frames = [load_single_gtfs_file(path) for path in paths]
    combined = pd.concat(frames, ignore_index=True)
    validate_required_columns(combined, REQUIRED_GTFS_COLUMNS, "GTFS-Realtime CSV")

    combined = combined[REQUIRED_GTFS_COLUMNS + ["source_file"]].copy()
    combined = normalize_realtime_types(combined)

    before_drop = len(combined)
    combined = combined.dropna(subset=["collection_time_utc", "delay_minutes", "delay_seconds", "route_id", "trip_id"]).copy()
    return combined, before_drop - len(combined), "daily_csv_fallback"

def load_gtfs_realtime_parquet(path, source_label):
    frame = pd.read_parquet(path)
    frame = clean_column_names(frame)
    validate_required_columns(frame, REQUIRED_GTFS_COLUMNS, "GTFS-Realtime Parquet")

    if "source_file" not in frame.columns:
        frame["source_file"] = path.name

    keep_columns = REQUIRED_GTFS_COLUMNS + [column for column in CANONICAL_PARQUET_COLUMNS if column in frame.columns]
    frame = frame[keep_columns].copy()
    frame = normalize_realtime_types(frame)

    before_drop = len(frame)
    frame = frame.dropna(subset=["collection_time_utc", "delay_minutes", "delay_seconds", "route_id", "trip_id"]).copy()
    return frame, before_drop - len(frame), source_label

def load_gtfs_realtime_dataset(input_dataset, parquet_path, csv_paths):
    parquet_error = None
    parquet_candidates = []

    if input_dataset == "auto":
        parquet_candidates = [
            (MODEL_BASELINE_PARQUET_PATH, "model_baseline_parquet"),
            (ALL_FILE_PARQUET_PATH, "all_file_parquet"),
        ]
    elif input_dataset in ["model_baseline", "all_file"]:
        label = "model_baseline_parquet" if input_dataset == "model_baseline" else "all_file_parquet"
        parquet_candidates = [(parquet_path, label)]

    for candidate_path, source_label in parquet_candidates:
        if candidate_path and candidate_path.exists():
            try:
                return load_gtfs_realtime_parquet(candidate_path, source_label)
            except Exception as error:
                parquet_error = error
                if input_dataset != "auto":
                    raise

    if csv_paths:
        if parquet_error is not None:
            print("Parquet load failed; using CSV fallback. Parquet error:", repr(parquet_error))
        return load_gtfs_realtime_files(csv_paths)

    if parquet_error is not None:
        raise parquet_error

    raise FileNotFoundError("No readable canonical Parquet input or CSV fallback files were available.")

# Load selected GTFS-Realtime dataset

gtfs_df, dropped_invalid_rows, input_source_used = load_gtfs_realtime_dataset(
    INPUT_DATASET,
    selected_parquet_path,
    selected_gtfs_files,
)

print("Input source used:", input_source_used)
print("Loaded GTFS-Realtime shape:", gtfs_df.shape)
print("Rows dropped during basic cleaning:", dropped_invalid_rows)
print("Source files represented:", gtfs_df["source_file"].nunique() if "source_file" in gtfs_df.columns else "not available")
print("Time range:", gtfs_df["collection_time_utc"].min(), "to", gtfs_df["collection_time_utc"].max())

for column in FINAL_STORAGE_COLUMNS:
    print(f"{column} present:", column in gtfs_df.columns)

gtfs_df.head()


Input source used: all_file_parquet
Loaded GTFS-Realtime shape: (3904432, 24)
Rows dropped during basic cleaning: 0
Source files represented: 31
Time range: 2026-05-06 09:34:55+00:00 to 2026-06-05 16:00:34+00:00
collection_day_status present: True
collection_coverage_hours present: True
is_partial_day present: True
is_model_baseline_day present: True
service_type present: True
route_display_name present: True
is_special_route present: True


,collection_time_utc,entity_id,trip_id,route_id,direction_id,start_time,start_date,timestamp,delay_seconds,is_deleted,delay_minutes,collection_date,source_file,collection_day_status,collection_coverage_hours,is_partial_day,is_model_baseline_day,agency_id,route_short_name,route_long_name,route_type,service_type,route_display_name,is_special_route
0,2026-05-06 09:34:55+00:00,902-98011-82800-1-cdd3d9f0,902-98011-82800-1-cdd3d9f0,MTIA-209,0,23:00:00,20260506,1778000407,0,False,0.000000,2026-05-06,gtfs_realtime_2026-05-06.csv,partial_or_interrupted,0.5,True,False,FGL,MTIA,MTIA,4,Ferry,MTIA,False
1,2026-05-06 09:34:55+00:00,902-98011-88200-1-cdd3d9f0,902-98011-88200-1-cdd3d9f0,MTIA-209,0,24:30:00,20260506,1778000413,0,False,0.000000,2026-05-06,gtfs_realtime_2026-05-06.csv,partial_or_interrupted,0.5,True,False,FGL,MTIA,MTIA,4,Ferry,MTIA,False
2,2026-05-06 09:34:55+00:00,246-850001-75660-2-5270112-3da0c47e,246-850001-75660-2-5270112-3da0c47e,EAST-201,0,21:01:00,20260506,1778060030,50,False,0.833333,2026-05-06,gtfs_realtime_2026-05-06.csv,partial_or_interrupted,0.5,True,False,AM,EAST,EAST,2,Train / Rail,EAST,False
3,2026-05-06 09:34:55+00:00,248-840042-73620-2-6159400-f7b76986,248-840042-73620-2-6159400-f7b76986,ONE-201,1,20:27:00,20260506,1778057120,17,False,0.283333,2026-05-06,gtfs_realtime_2026-05-06.csv,partial_or_interrupted,0.5,True,False,AM,ONE,ONE,2,Train / Rail,ONE,False
4,2026-05-06 09:34:55+00:00,248-840085-78780-2-6164400-25f91d7f,248-840085-78780-2-6164400-25f91d7f,ONE-201,0,21:53:00,20260506,1778051978,0,False,0.000000,2026-05-06,gtfs_realtime_2026-05-06.csv,partial_or_interrupted,0.5,True,False,AM,ONE,ONE,2,Train / Rail,ONE,False


In [23]:
# Basic GTFS-Realtime validation checks

print("Missing values by column:")
print(gtfs_df.isna().sum())

print("\nUnique routes:", gtfs_df["route_id"].nunique())
print("Unique trips:", gtfs_df["trip_id"].nunique())
print("Unique collection timestamps:", gtfs_df["collection_time_utc"].nunique())

print("\nDelay minutes summary:")
print(gtfs_df["delay_minutes"].describe())

print("\nDuplicate rows:", gtfs_df.duplicated().sum())


Missing values by column:
collection_time_utc               0
entity_id                         0
trip_id                           0
route_id                          0
direction_id                      0
start_time                        0
start_date                        0
timestamp                         0
delay_seconds                     0
is_deleted                        0
delay_minutes                     0
collection_date                   0
source_file                       0
collection_day_status             0
collection_coverage_hours         0
is_partial_day                    0
is_model_baseline_day             0
agency_id                    103964
route_short_name             103964
route_long_name              103964
route_type                   103964
service_type                      0
route_display_name                0
is_special_route                  0
dtype: int64

Unique routes: 524
Unique trips: 34560
Unique collection timestamps: 3499

Delay minutes summary

### Sanity Check - GTFS-Realtime Cleaning

Expected result: `delay_seconds` and `delay_minutes` are numeric, `collection_time_utc` is timezone-aware UTC, duplicate header rows are gone, and the remaining records represent real Auckland routes and trips.


## Step 3 - Filter Delay Outliers

Very large negative or positive delays can distort validation summaries. This notebook keeps delays from -60 to +120 minutes. Those bounds are intentionally visible so they can be defended or adjusted in the capstone report.


In [24]:
# Outlier handling

MIN_REASONABLE_DELAY_MINUTES = -60
MAX_REASONABLE_DELAY_MINUTES = 120

outlier_mask = gtfs_df["delay_minutes"].between(MIN_REASONABLE_DELAY_MINUTES, MAX_REASONABLE_DELAY_MINUTES, inclusive="both")
gtfs_clean = gtfs_df.loc[outlier_mask].copy()
outlier_rows_removed = len(gtfs_df) - len(gtfs_clean)

print("Original rows:", len(gtfs_df))
print("Rows after outlier filter:", len(gtfs_clean))
print("Outlier rows removed:", outlier_rows_removed)
print("Outlier removal rate:", round(outlier_rows_removed / len(gtfs_df) * 100, 2), "%")

print("\nCleaned delay summary:")
print(gtfs_clean["delay_minutes"].describe())


Original rows: 3904432
Rows after outlier filter: 3903290
Outlier rows removed: 1142
Outlier removal rate: 0.03 %

Cleaned delay summary:
count    3.903290e+06
mean    -4.622798e-01
std      5.437075e+00
min     -6.000000e+01
25%     -3.033333e+00
50%     -8.333333e-02
75%      1.483333e+00
max      1.199500e+02
Name: delay_minutes, dtype: float64


### Sanity Check - Outlier Filter

Expected result: only a small share of records should be removed. If many records are removed, the raw feed or delay calculation should be reviewed before using the data for decision support.


## Step 4 - Create Temporal Features

Temporal features make the validation useful for operations. They allow the dashboard and summaries to compare delay patterns by hour, weekday, and day of month.


In [25]:
# Temporal feature engineering

gtfs_clean["trip_hour"] = gtfs_clean["collection_time_utc"].dt.hour
gtfs_clean["weekday"] = gtfs_clean["collection_time_utc"].dt.dayofweek
gtfs_clean["day_of_month"] = gtfs_clean["collection_time_utc"].dt.day
gtfs_clean["collection_date"] = gtfs_clean["collection_time_utc"].dt.date
gtfs_clean["delay_abs_minutes"] = gtfs_clean["delay_minutes"].abs()

print("Feature-engineered shape:", gtfs_clean.shape)
print("Trip hour range:", gtfs_clean["trip_hour"].min(), "to", gtfs_clean["trip_hour"].max())
print("Weekdays represented:", sorted(gtfs_clean["weekday"].dropna().unique().tolist()))

gtfs_clean[["collection_time_utc", "collection_date", "route_id", "trip_id", "delay_minutes", "trip_hour", "weekday", "day_of_month"]].head()


Feature-engineered shape: (3903290, 28)
Trip hour range: 0 to 23
Weekdays represented: [0, 1, 2, 3, 4, 5, 6]


,collection_time_utc,collection_date,route_id,trip_id,delay_minutes,trip_hour,weekday,day_of_month
0,2026-05-06 09:34:55+00:00,2026-05-06,MTIA-209,902-98011-82800-1-cdd3d9f0,0.000000,9,2,6
1,2026-05-06 09:34:55+00:00,2026-05-06,MTIA-209,902-98011-88200-1-cdd3d9f0,0.000000,9,2,6
2,2026-05-06 09:34:55+00:00,2026-05-06,EAST-201,246-850001-75660-2-5270112-3da0c47e,0.833333,9,2,6
3,2026-05-06 09:34:55+00:00,2026-05-06,ONE-201,248-840042-73620-2-6159400-f7b76986,0.283333,9,2,6
4,2026-05-06 09:34:55+00:00,2026-05-06,ONE-201,248-840085-78780-2-6164400-25f91d7f,0.000000,9,2,6


### Sanity Check - Temporal Features

Expected result: `trip_hour`, `weekday`, and `day_of_month` are populated for every retained GTFS-Realtime record.


## Step 5 - Validate Route Metadata, Service Fields, and Corridor Names

The canonical Parquet datasets already include GTFS Static route metadata, service type, display names, and special-route flags from the Data Storage module. Notebook 09 preserves those fields. It also derives a clearer `route_corridor_name` from GTFS Static `trips.txt` headsigns so route summaries are understandable to non-technical readers. If the notebook is run from raw CSV fallback, it still merges GTFS Static route metadata here so the validation flow remains reproducible.


In [26]:
# Load GTFS Static route metadata and route corridor names for CSV fallback, audit checks, and readable summaries

routes_path = GTFS_STATIC_DIR / "routes.txt"
trips_path = GTFS_STATIC_DIR / "trips.txt"

if not routes_path.exists():
    raise FileNotFoundError(f"GTFS Static routes file not found: {routes_path}")
if not trips_path.exists():
    raise FileNotFoundError(f"GTFS Static trips file not found: {trips_path}")

routes_df = pd.read_csv(routes_path, dtype=str)
routes_df = clean_column_names(routes_df)

required_route_columns = ["route_id", "route_short_name", "route_long_name", "route_type", "agency_id"]
missing_route_columns = sorted(set(required_route_columns) - set(routes_df.columns))
if missing_route_columns:
    raise ValueError(f"Missing GTFS Static route columns: {missing_route_columns}")

routes_lookup = routes_df[required_route_columns].drop_duplicates(subset=["route_id"]).copy()

trips_df = pd.read_csv(trips_path, dtype=str, usecols=lambda column: column.strip().lower() in ["route_id", "trip_headsign"])
trips_df = clean_column_names(trips_df)

if "trip_headsign" not in trips_df.columns:
    raise ValueError("GTFS Static trips.txt must include trip_headsign to derive route corridor names.")

def build_route_corridor_label(group):
    route_id = str(group.name)
    route_short_name = routes_lookup.loc[routes_lookup["route_id"].astype(str).eq(route_id), "route_short_name"]
    route_label = route_short_name.iloc[0] if len(route_short_name) and pd.notna(route_short_name.iloc[0]) else route_id

    headsigns = group.dropna().astype(str).str.strip()
    headsigns = headsigns[headsigns.ne("")].drop_duplicates().tolist()

    endpoints = []
    for headsign in headsigns:
        normalized = headsign.replace(" to ", " To ")
        if " To " in normalized:
            parts = [part.strip() for part in normalized.split(" To ", 1)]
            endpoints.extend([part for part in parts if part])
        else:
            endpoints.append(normalized)

    unique_endpoints = []
    for endpoint in endpoints:
        if endpoint not in unique_endpoints:
            unique_endpoints.append(endpoint)

    if len(unique_endpoints) >= 2:
        corridor = " / ".join(unique_endpoints[:2])
    elif len(unique_endpoints) == 1:
        corridor = unique_endpoints[0]
    else:
        corridor = route_label

    return f"{route_label} - {corridor}" if corridor and corridor != route_label else str(route_label)

route_corridor_lookup = (
    trips_df.groupby("route_id")["trip_headsign"]
    .apply(build_route_corridor_label)
    .reset_index(name="route_corridor_name")
)

routes_lookup = routes_lookup.merge(route_corridor_lookup, on="route_id", how="left")
routes_lookup["route_corridor_name"] = routes_lookup["route_corridor_name"].fillna(routes_lookup["route_short_name"]).fillna(routes_lookup["route_id"])

print("Routes shape:", routes_df.shape)
print("Route lookup shape:", routes_lookup.shape)
print("Route corridor names available:", routes_lookup["route_corridor_name"].notna().sum())
print("Example corridor for 128-202:")
print(routes_lookup.loc[routes_lookup["route_id"].eq("128-202"), ["route_id", "route_short_name", "route_corridor_name"]])
routes_lookup.head()


Routes shape: (220, 11)
Route lookup shape: (220, 6)
Route corridor names available: 220
Example corridor for 128-202:
   route_id route_short_name                         route_corridor_name
20  128-202              128  128 - Hibiscus Coast Station / Helensville


,route_id,route_short_name,route_long_name,route_type,agency_id,route_corridor_name
0,101-202,101,101,3,NZB,101 - Pt Chevalier / Auckland University Via J...
1,STH-201,STH,STH,2,AM,"STH - Brit 4 / Pukekohe 1 Via NKT 4, OHU 3, Pa..."
2,TIRI-240,TIRI,TIRI,4,EXPNZ,TIRI - Gulf Harbour / Tiritiri Matangi Island
3,TMK-202,TMK,TMK,3,NZB,TMK - Glen Innes / Britomart Via St Heliers An...
4,WEST-201,WEST,WEST,2,AM,WEST - Brit 2 / Swanson 1 Via Newmarket 1


In [27]:
# Validate or add route metadata

route_metadata_columns = ["route_short_name", "route_long_name", "route_type", "agency_id"]
parquet_has_route_metadata = all(column in gtfs_clean.columns for column in route_metadata_columns)

if parquet_has_route_metadata:
    gtfs_routes = gtfs_clean.copy()
    route_metadata_source = "canonical_parquet"
else:
    gtfs_routes = gtfs_clean.merge(routes_lookup, on="route_id", how="left", validate="many_to_one")
    route_metadata_source = "gtfs_static_merge"

if "service_type" not in gtfs_routes.columns:
    route_type_map = {
        "2": "Train / Rail",
        "3": "Bus",
        "4": "Ferry",
        2: "Train / Rail",
        3: "Bus",
        4: "Ferry",
    }
    gtfs_routes["service_type"] = gtfs_routes["route_type"].map(route_type_map).fillna("Unknown / unmatched route")

if "route_display_name" not in gtfs_routes.columns:
    display_from_static = gtfs_routes["route_short_name"].fillna(gtfs_routes["route_long_name"])
    gtfs_routes["route_display_name"] = display_from_static.fillna(gtfs_routes["route_id"])

if "route_corridor_name" not in gtfs_routes.columns:
    gtfs_routes = gtfs_routes.merge(routes_lookup[["route_id", "route_corridor_name"]], on="route_id", how="left", validate="many_to_one")
else:
    gtfs_routes["route_corridor_name"] = gtfs_routes["route_corridor_name"].replace("", np.nan)

gtfs_routes["route_corridor_name"] = (
    gtfs_routes["route_corridor_name"]
    .fillna(gtfs_routes["route_display_name"])
    .fillna(gtfs_routes["route_short_name"])
    .fillna(gtfs_routes["route_id"])
)

if "is_special_route" not in gtfs_routes.columns:
    gtfs_routes["is_special_route"] = gtfs_routes["route_id"].astype(str).str.startswith("S")
else:
    gtfs_routes["is_special_route"] = gtfs_routes["is_special_route"].fillna(False).astype(bool)

matched_routes = gtfs_routes["route_short_name"].notna().sum()
total_route_records = len(gtfs_routes)
route_match_rate = matched_routes / total_route_records if total_route_records else np.nan
s_prefix_rows = gtfs_routes["route_id"].astype(str).str.startswith("S", na=False).sum()
s_prefix_flagged = gtfs_routes.loc[gtfs_routes["route_id"].astype(str).str.startswith("S", na=False), "is_special_route"].sum()

print("Route metadata source:", route_metadata_source)
print("Realtime GTFS before route validation:", gtfs_clean.shape)
print("Realtime GTFS after route validation:", gtfs_routes.shape)
print("Matched route records:", matched_routes)
print("Route match rate:", round(route_match_rate * 100, 2), "%")
print("S-prefix route rows retained:", int(s_prefix_rows))
print("S-prefix route rows flagged special:", int(s_prefix_flagged))
print("Example 128-202 corridor:", gtfs_routes.loc[gtfs_routes["route_id"].astype(str).eq("128-202"), "route_corridor_name"].drop_duplicates().head().tolist())

print("\nMissing route/service metadata:")
print(gtfs_routes[["route_short_name", "route_long_name", "route_type", "agency_id", "service_type", "route_display_name", "route_corridor_name", "is_special_route"]].isna().sum())

gtfs_routes.head()


Route metadata source: canonical_parquet
Realtime GTFS before route validation: (3903290, 28)
Realtime GTFS after route validation: (3903290, 29)
Matched route records: 3799361
Route match rate: 97.34 %
S-prefix route rows retained: 227918
S-prefix route rows flagged special: 227918
Example 128-202 corridor: ['128 - Hibiscus Coast Station / Helensville']

Missing route/service metadata:
route_short_name       103929
route_long_name        103929
route_type             103929
agency_id              103929
service_type                0
route_display_name          0
route_corridor_name         0
is_special_route            0
dtype: int64


,collection_time_utc,entity_id,trip_id,route_id,direction_id,start_time,start_date,timestamp,delay_seconds,is_deleted,delay_minutes,collection_date,source_file,collection_day_status,collection_coverage_hours,is_partial_day,is_model_baseline_day,agency_id,route_short_name,route_long_name,route_type,service_type,route_display_name,is_special_route,trip_hour,weekday,day_of_month,delay_abs_minutes,route_corridor_name
0,2026-05-06 09:34:55+00:00,902-98011-82800-1-cdd3d9f0,902-98011-82800-1-cdd3d9f0,MTIA-209,0,23:00:00,20260506,1778000407,0,False,0.000000,2026-05-06,gtfs_realtime_2026-05-06.csv,partial_or_interrupted,0.5,True,False,FGL,MTIA,MTIA,4,Ferry,MTIA,False,9,2,6,0.000000,MTIA - Waiheke Island 1 / Downtown Pier 11
1,2026-05-06 09:34:55+00:00,902-98011-88200-1-cdd3d9f0,902-98011-88200-1-cdd3d9f0,MTIA-209,0,24:30:00,20260506,1778000413,0,False,0.000000,2026-05-06,gtfs_realtime_2026-05-06.csv,partial_or_interrupted,0.5,True,False,FGL,MTIA,MTIA,4,Ferry,MTIA,False,9,2,6,0.000000,MTIA - Waiheke Island 1 / Downtown Pier 11
2,2026-05-06 09:34:55+00:00,246-850001-75660-2-5270112-3da0c47e,246-850001-75660-2-5270112-3da0c47e,EAST-201,0,21:01:00,20260506,1778060030,50,False,0.833333,2026-05-06,gtfs_realtime_2026-05-06.csv,partial_or_interrupted,0.5,True,False,AM,EAST,EAST,2,Train / Rail,EAST,False,9,2,6,0.833333,EAST - Manukau 1 / Brit 1 Via Panmure
3,2026-05-06 09:34:55+00:00,248-840042-73620-2-6159400-f7b76986,248-840042-73620-2-6159400-f7b76986,ONE-201,1,20:27:00,20260506,1778057120,17,False,0.283333,2026-05-06,gtfs_realtime_2026-05-06.csv,partial_or_interrupted,0.5,True,False,AM,ONE,ONE,2,Train / Rail,ONE,False,9,2,6,0.283333,ONE - Newmarket 4 / Onehunga
4,2026-05-06 09:34:55+00:00,248-840085-78780-2-6164400-25f91d7f,248-840085-78780-2-6164400-25f91d7f,ONE-201,0,21:53:00,20260506,1778051978,0,False,0.000000,2026-05-06,gtfs_realtime_2026-05-06.csv,partial_or_interrupted,0.5,True,False,AM,ONE,ONE,2,Train / Rail,ONE,False,9,2,6,0.000000,ONE - Newmarket 4 / Onehunga


### Sanity Check - Route Metadata and Special Services

Expected result: the route match rate should be high, final storage fields such as `service_type`, `route_display_name`, and `is_special_route` should be present, and S-prefix routes should remain in the dataset. S-prefix routes are flagged as special services so they can be included in full coverage views but excluded from default top-delayed route rankings.


## Step 5A - Route Coverage Limitation Check

The GTFS Static merge can legitimately miss some realtime route identifiers when the realtime feed contains school, special, or operational variants that are not present in the static `routes.txt` snapshot. This section quantifies those unmatched records so the route match rate can be reported transparently.


In [28]:
# Inspect unmatched realtime route IDs after the GTFS Static merge

unmatched_routes = gtfs_routes[gtfs_routes["route_short_name"].isna()].copy()

unmatched_route_summary = (
    unmatched_routes
    .groupby("route_id", as_index=False)
    .agg(
        records=("trip_id", "size"),
        unique_trips=("trip_id", "nunique"),
        avg_delay_minutes=("delay_minutes", "mean"),
        max_delay_minutes=("delay_minutes", "max"),
    )
    .sort_values(["records", "avg_delay_minutes"], ascending=[False, False])
)

unmatched_route_summary["avg_delay_minutes"] = unmatched_route_summary["avg_delay_minutes"].round(2)
unmatched_route_summary["route_prefix"] = unmatched_route_summary["route_id"].str.extract(r"^([A-Za-z]+)", expand=False).fillna("numeric_or_other")

unmatched_by_source_file = (
    unmatched_routes
    .groupby("source_file", as_index=False)
    .size()
    .rename(columns={"size": "unmatched_records"})
)

print("Unmatched records:", len(unmatched_routes))
print("Unmatched unique routes:", unmatched_routes["route_id"].nunique())
print("Unmatched share:", round(len(unmatched_routes) / len(gtfs_routes) * 100, 2), "%")

print("\nUnmatched route prefix counts:")
print(unmatched_route_summary["route_prefix"].value_counts())

print("\nTop unmatched routes:")
unmatched_route_summary.head(20)


Unmatched records: 103929
Unmatched unique routes: 317
Unmatched share: 2.66 %

Unmatched route prefix counts:
S                   316
numeric_or_other      1
Name: route_prefix, dtype: int64

Top unmatched routes:


,route_id,records,unique_trips,avg_delay_minutes,max_delay_minutes,route_prefix
0,327-203,2660,95,-2.98,58.016667,numeric_or_other
287,S510-202,1038,5,-3.17,13.583333,S
312,S547-202,962,5,-0.01,8.983333,S
59,S017F-203,954,6,-2.56,14.116667,S
138,S050B-203,892,5,-2.44,35.750000,S
235,S413-202,843,4,-1.38,10.616667,S
239,S417-202,807,4,2.20,24.683333,S
19,S006C-203,797,4,-0.70,47.583333,S
307,S542-202,717,4,-1.27,14.850000,S
22,S007F-203,706,8,2.43,17.883333,S


### Sanity Check - Unmatched Routes

Expected result: unmatched records should be explainable, not silently removed. In the canonical Parquet datasets, unmatched or special-service rows are retained with `service_type` and `route_display_name` fallbacks so dashboard and audit views can still show them clearly.


## Step 6 - Create Delay-Risk Categories

Delay-risk categories translate numeric delay values into operational severity levels. This keeps the decision engine understandable for non-technical readers.


In [29]:
# Delay-risk category logic

RISK_ORDER = ["Low", "Medium", "High", "Severe"]

def classify_realtime_delay_risk(delay_minutes):
    if delay_minutes <= 5:
        return "Low"
    if delay_minutes <= 15:
        return "Medium"
    if delay_minutes <= 25:
        return "High"
    return "Severe"

gtfs_routes["delay_risk"] = gtfs_routes["delay_abs_minutes"].apply(classify_realtime_delay_risk)
gtfs_routes["delay_risk"] = pd.Categorical(gtfs_routes["delay_risk"], categories=RISK_ORDER, ordered=True)

risk_summary = gtfs_routes["delay_risk"].value_counts(sort=False).rename_axis("delay_risk").reset_index(name="records")
risk_summary["share_pct"] = (risk_summary["records"] / len(gtfs_routes) * 100).round(2)
risk_summary


,delay_risk,records,share_pct
0,Low,3022730,77.44
1,Medium,819796,21.00
2,High,45394,1.16
3,Severe,15370,0.39


### Sanity Check - Delay Risk

Expected result: the risk table should include all four categories. If the severe category is large, inspect whether the route, collection period, or outlier bounds explain the pattern.


## Step 7 - Fetch Open-Meteo Hourly Weather

Weather is aligned to GTFS collection timestamps by hour. The request uses UTC so the weather timestamps match `collection_time_utc` directly. If network access is unavailable during reruns, use a documented local Open-Meteo extract with the same hourly columns and date range instead of calling the API.


In [30]:
# Open-Meteo hourly weather request

latitude = -36.8485
longitude = 174.7633

weather_start_date = gtfs_routes["collection_time_utc"].min().strftime("%Y-%m-%d")
weather_end_date = gtfs_routes["collection_time_utc"].max().strftime("%Y-%m-%d")

weather_url = "https://archive-api.open-meteo.com/v1/archive"
weather_params = {
    "latitude": latitude,
    "longitude": longitude,
    "start_date": weather_start_date,
    "end_date": weather_end_date,
    "hourly": ",".join([
        "temperature_2m",
        "precipitation",
        "rain",
        "relative_humidity_2m",
        "wind_speed_10m",
    ]),
    "timezone": "UTC",
}

print("Weather request date range:", weather_start_date, "to", weather_end_date)

weather_json = None
weather_df = pd.DataFrame()
weather_source = "Open-Meteo API"

try:
    weather_request_url = f"{weather_url}?{urlencode(weather_params)}"
    with urlopen(weather_request_url, timeout=30) as response:
        weather_status = response.status
        weather_json = json.loads(response.read().decode("utf-8"))

    print("Weather API status:", weather_status)

    if "hourly" not in weather_json:
        raise ValueError("Open-Meteo response does not contain an hourly weather table.")

except Exception as error:
    weather_source = "local decision_engine_output.csv fallback"
    print("Weather API unavailable; using local weather fallback.")
    print("Weather fallback reason:", type(error).__name__, str(error))

    local_weather_path = DATA_PROCESSED / "decision_engine_output.csv"
    local_weather_columns = [
        "collection_time_utc",
        "temperature_2m",
        "precipitation",
        "rain",
        "relative_humidity_2m",
        "wind_speed_10m",
    ]

    if not local_weather_path.exists():
        raise FileNotFoundError(
            "Open-Meteo API was unavailable and local decision_engine_output.csv was not found."
        )

    weather_df = pd.read_csv(
        local_weather_path,
        usecols=local_weather_columns,
        low_memory=False,
    )
    weather_df["weather_hour"] = parse_utc_datetime(weather_df["collection_time_utc"]).dt.floor("h")
    weather_df = (
        weather_df
        .dropna(subset=["weather_hour"])
        .drop_duplicates(subset=["weather_hour"])
        [[
            "weather_hour",
            "temperature_2m",
            "precipitation",
            "rain",
            "relative_humidity_2m",
            "wind_speed_10m",
        ]]
        .copy()
    )

print("Weather source:", weather_source)


Weather request date range: 2026-05-06 to 2026-06-05
Weather API status: 200
Weather source: Open-Meteo API


In [31]:
# Convert weather response to dataframe

if weather_df.empty:
    weather_df = pd.DataFrame(weather_json["hourly"])
    weather_df = clean_column_names(weather_df)

    if "time" not in weather_df.columns:
        raise ValueError("Weather data is missing the time column.")

    weather_df["weather_hour"] = parse_utc_datetime(weather_df["time"])
else:
    weather_df = clean_column_names(weather_df)

weather_df = (
    weather_df
    .dropna(subset=["weather_hour"])
    .drop_duplicates(subset=["weather_hour"])
    .copy()
)

print("Weather dataset shape:", weather_df.shape)
print("Weather time range:", weather_df["weather_hour"].min(), "to", weather_df["weather_hour"].max())
weather_df.head()


Weather dataset shape: (744, 7)
Weather time range: 2026-05-06 00:00:00+00:00 to 2026-06-05 23:00:00+00:00


,time,temperature_2m,precipitation,rain,relative_humidity_2m,wind_speed_10m,weather_hour
0,2026-05-06T00:00,18.4,0.2,0.2,83,12.9,2026-05-06 00:00:00+00:00
1,2026-05-06T01:00,19.0,0.0,0.0,79,13.5,2026-05-06 01:00:00+00:00
2,2026-05-06T02:00,19.3,0.1,0.1,76,13.9,2026-05-06 02:00:00+00:00
3,2026-05-06T03:00,19.2,0.0,0.0,75,13.9,2026-05-06 03:00:00+00:00
4,2026-05-06T04:00,18.8,0.0,0.0,77,14.4,2026-05-06 04:00:00+00:00


In [32]:
# Align GTFS timestamps to hourly weather timestamps

gtfs_routes["weather_hour"] = gtfs_routes["collection_time_utc"].dt.floor("h")

print("Realtime weather-hour range:", gtfs_routes["weather_hour"].min(), "to", gtfs_routes["weather_hour"].max())
print("Weather table range:", weather_df["weather_hour"].min(), "to", weather_df["weather_hour"].max())

print("\nGTFS hourly timestamp sample:")
print(gtfs_routes["weather_hour"].head())

print("\nWeather hourly timestamp sample:")
print(weather_df["weather_hour"].head())


Realtime weather-hour range: 2026-05-06 09:00:00+00:00 to 2026-06-05 16:00:00+00:00
Weather table range: 2026-05-06 00:00:00+00:00 to 2026-06-05 23:00:00+00:00

GTFS hourly timestamp sample:
0   2026-05-06 09:00:00+00:00
1   2026-05-06 09:00:00+00:00
2   2026-05-06 09:00:00+00:00
3   2026-05-06 09:00:00+00:00
4   2026-05-06 09:00:00+00:00
Name: weather_hour, dtype: datetime64[ns, UTC]

Weather hourly timestamp sample:
0   2026-05-06 00:00:00+00:00
1   2026-05-06 01:00:00+00:00
2   2026-05-06 02:00:00+00:00
3   2026-05-06 03:00:00+00:00
4   2026-05-06 04:00:00+00:00
Name: weather_hour, dtype: datetime64[ns, UTC]


In [33]:
# Merge realtime GTFS records with hourly weather data

weather_columns = ["weather_hour", "temperature_2m", "precipitation", "rain", "relative_humidity_2m", "wind_speed_10m"]
weather_value_columns = [column for column in weather_columns if column != "weather_hour"]
missing_weather_columns = sorted(set(weather_columns) - set(weather_df.columns))
if missing_weather_columns:
    raise ValueError(f"Missing weather columns: {missing_weather_columns}")

weather_lookup = (
    weather_df[weather_columns]
    .drop_duplicates(subset=["weather_hour"])
    .set_index("weather_hour")
)

# Use hourly lookup mapping instead of a full dataframe merge to reduce memory use on multi-million-row datasets.
gtfs_weather = gtfs_routes.copy(deep=False)
for column in weather_value_columns:
    gtfs_weather[column] = gtfs_weather["weather_hour"].map(weather_lookup[column])
    gtfs_weather[column] = pd.to_numeric(gtfs_weather[column], errors="coerce", downcast="float")

matched_weather = gtfs_weather["temperature_2m"].notna().sum()
total_weather_records = len(gtfs_weather)
weather_match_rate = matched_weather / total_weather_records if total_weather_records else np.nan

print("Integrated GTFS + weather shape:", gtfs_weather.shape)
print("Weather matched records:", matched_weather)
print("Weather match rate:", round(weather_match_rate * 100, 2), "%")

print("\nMissing weather values:")
print(gtfs_weather[weather_value_columns].isna().sum())

gtfs_weather.head()


Integrated GTFS + weather shape: (3903290, 36)
Weather matched records: 3903290
Weather match rate: 100.0 %

Missing weather values:
temperature_2m          0
precipitation           0
rain                    0
relative_humidity_2m    0
wind_speed_10m          0
dtype: int64


,collection_time_utc,entity_id,trip_id,route_id,direction_id,start_time,start_date,timestamp,delay_seconds,is_deleted,delay_minutes,collection_date,source_file,collection_day_status,collection_coverage_hours,is_partial_day,is_model_baseline_day,agency_id,route_short_name,route_long_name,route_type,service_type,route_display_name,is_special_route,trip_hour,weekday,day_of_month,delay_abs_minutes,route_corridor_name,delay_risk,weather_hour,temperature_2m,precipitation,rain,relative_humidity_2m,wind_speed_10m
0,2026-05-06 09:34:55+00:00,902-98011-82800-1-cdd3d9f0,902-98011-82800-1-cdd3d9f0,MTIA-209,0,23:00:00,20260506,1778000407,0,False,0.000000,2026-05-06,gtfs_realtime_2026-05-06.csv,partial_or_interrupted,0.5,True,False,FGL,MTIA,MTIA,4,Ferry,MTIA,False,9,2,6,0.000000,MTIA - Waiheke Island 1 / Downtown Pier 11,Low,2026-05-06 09:00:00+00:00,17.4,0.0,0.0,86.0,13.1
1,2026-05-06 09:34:55+00:00,902-98011-88200-1-cdd3d9f0,902-98011-88200-1-cdd3d9f0,MTIA-209,0,24:30:00,20260506,1778000413,0,False,0.000000,2026-05-06,gtfs_realtime_2026-05-06.csv,partial_or_interrupted,0.5,True,False,FGL,MTIA,MTIA,4,Ferry,MTIA,False,9,2,6,0.000000,MTIA - Waiheke Island 1 / Downtown Pier 11,Low,2026-05-06 09:00:00+00:00,17.4,0.0,0.0,86.0,13.1
2,2026-05-06 09:34:55+00:00,246-850001-75660-2-5270112-3da0c47e,246-850001-75660-2-5270112-3da0c47e,EAST-201,0,21:01:00,20260506,1778060030,50,False,0.833333,2026-05-06,gtfs_realtime_2026-05-06.csv,partial_or_interrupted,0.5,True,False,AM,EAST,EAST,2,Train / Rail,EAST,False,9,2,6,0.833333,EAST - Manukau 1 / Brit 1 Via Panmure,Low,2026-05-06 09:00:00+00:00,17.4,0.0,0.0,86.0,13.1
3,2026-05-06 09:34:55+00:00,248-840042-73620-2-6159400-f7b76986,248-840042-73620-2-6159400-f7b76986,ONE-201,1,20:27:00,20260506,1778057120,17,False,0.283333,2026-05-06,gtfs_realtime_2026-05-06.csv,partial_or_interrupted,0.5,True,False,AM,ONE,ONE,2,Train / Rail,ONE,False,9,2,6,0.283333,ONE - Newmarket 4 / Onehunga,Low,2026-05-06 09:00:00+00:00,17.4,0.0,0.0,86.0,13.1
4,2026-05-06 09:34:55+00:00,248-840085-78780-2-6164400-25f91d7f,248-840085-78780-2-6164400-25f91d7f,ONE-201,0,21:53:00,20260506,1778051978,0,False,0.000000,2026-05-06,gtfs_realtime_2026-05-06.csv,partial_or_interrupted,0.5,True,False,AM,ONE,ONE,2,Train / Rail,ONE,False,9,2,6,0.000000,ONE - Newmarket 4 / Onehunga,Low,2026-05-06 09:00:00+00:00,17.4,0.0,0.0,86.0,13.1


### Sanity Check - Weather Integration

Expected result: the weather match rate should be close to 100%. If it is low, check timezone handling first, then confirm the Open-Meteo date range covers the GTFS-Realtime collection period.


## Step 8 - Operational Recommendations

The recommendation rules translate delay-risk categories into plain-language transport operations actions. These are decision-support recommendations, not automated control decisions.


In [34]:
# Operational recommendation logic

def generate_realtime_recommendation(risk_level):
    risk_level = str(risk_level)
    if risk_level == "Low":
        return "No operational action required"
    if risk_level == "Medium":
        return "Monitor route conditions"
    if risk_level == "High":
        return "Adjust service headway"
    return "Deploy standby bus or supervisor review"

gtfs_weather["recommended_action"] = gtfs_weather["delay_risk"].apply(generate_realtime_recommendation)

recommendation_summary = gtfs_weather["recommended_action"].value_counts().rename_axis("recommended_action").reset_index(name="records")
recommendation_summary["share_pct"] = (recommendation_summary["records"] / len(gtfs_weather) * 100).round(2)
recommendation_summary


,recommended_action,records,share_pct
0,No operational action required,3022730,77.44
1,Monitor route conditions,819796,21.00
2,Adjust service headway,45394,1.16
3,Deploy standby bus or supervisor review,15370,0.39


In [35]:
# Dashboard-ready decision engine table held in memory only

base_decision_columns = [
    "collection_time_utc", "collection_date", "collection_day_status", "collection_coverage_hours",
    "is_partial_day", "is_model_baseline_day", "route_id", "route_short_name", "route_long_name",
    "route_type", "agency_id", "service_type", "route_display_name", "route_corridor_name", "is_special_route",
    "trip_id", "direction_id", "delay_seconds", "delay_minutes", "delay_risk", "recommended_action",
    "trip_hour", "weekday", "day_of_month", "temperature_2m", "precipitation", "rain",
    "relative_humidity_2m", "wind_speed_10m", "source_file",
]

available_decision_columns = [column for column in base_decision_columns if column in gtfs_weather.columns]
missing_decision_columns = [column for column in base_decision_columns if column not in gtfs_weather.columns]

decision_engine_output = gtfs_weather[available_decision_columns].copy()

print("Decision engine output shape:", decision_engine_output.shape)
print("Missing optional decision output columns:", missing_decision_columns)
decision_engine_output.head()


Decision engine output shape: (3903290, 30)
Missing optional decision output columns: []


,collection_time_utc,collection_date,collection_day_status,collection_coverage_hours,is_partial_day,is_model_baseline_day,route_id,route_short_name,route_long_name,route_type,agency_id,service_type,route_display_name,route_corridor_name,is_special_route,trip_id,direction_id,delay_seconds,delay_minutes,delay_risk,recommended_action,trip_hour,weekday,day_of_month,temperature_2m,precipitation,rain,relative_humidity_2m,wind_speed_10m,source_file
0,2026-05-06 09:34:55+00:00,2026-05-06,partial_or_interrupted,0.5,True,False,MTIA-209,MTIA,MTIA,4,FGL,Ferry,MTIA,MTIA - Waiheke Island 1 / Downtown Pier 11,False,902-98011-82800-1-cdd3d9f0,0,0,0.000000,Low,No operational action required,9,2,6,17.4,0.0,0.0,86.0,13.1,gtfs_realtime_2026-05-06.csv
1,2026-05-06 09:34:55+00:00,2026-05-06,partial_or_interrupted,0.5,True,False,MTIA-209,MTIA,MTIA,4,FGL,Ferry,MTIA,MTIA - Waiheke Island 1 / Downtown Pier 11,False,902-98011-88200-1-cdd3d9f0,0,0,0.000000,Low,No operational action required,9,2,6,17.4,0.0,0.0,86.0,13.1,gtfs_realtime_2026-05-06.csv
2,2026-05-06 09:34:55+00:00,2026-05-06,partial_or_interrupted,0.5,True,False,EAST-201,EAST,EAST,2,AM,Train / Rail,EAST,EAST - Manukau 1 / Brit 1 Via Panmure,False,246-850001-75660-2-5270112-3da0c47e,0,50,0.833333,Low,No operational action required,9,2,6,17.4,0.0,0.0,86.0,13.1,gtfs_realtime_2026-05-06.csv
3,2026-05-06 09:34:55+00:00,2026-05-06,partial_or_interrupted,0.5,True,False,ONE-201,ONE,ONE,2,AM,Train / Rail,ONE,ONE - Newmarket 4 / Onehunga,False,248-840042-73620-2-6159400-f7b76986,1,17,0.283333,Low,No operational action required,9,2,6,17.4,0.0,0.0,86.0,13.1,gtfs_realtime_2026-05-06.csv
4,2026-05-06 09:34:55+00:00,2026-05-06,partial_or_interrupted,0.5,True,False,ONE-201,ONE,ONE,2,AM,Train / Rail,ONE,ONE - Newmarket 4 / Onehunga,False,248-840085-78780-2-6164400-25f91d7f,0,0,0.000000,Low,No operational action required,9,2,6,17.4,0.0,0.0,86.0,13.1,gtfs_realtime_2026-05-06.csv


### Sanity Check - Recommendations

Expected result: every retained record should have a delay-risk category and a recommended action. Recommendations are intentionally simple so they can be explained in the final report and dashboard.


## Step 9 - Dashboard Summary Tables

These tables are prepared in memory for Streamlit/dashboard use. They are written to disk only when `WRITE_OUTPUTS` is deliberately set to `True` for a reproducible output refresh.


In [36]:
# Dashboard-ready summaries held in memory only

daily_summary = decision_engine_output.groupby("collection_date", as_index=False).agg(
    records=("trip_id", "size"),
    unique_routes=("route_id", "nunique"),
    unique_trips=("trip_id", "nunique"),
    avg_delay_minutes=("delay_minutes", "mean"),
    min_delay_minutes=("delay_minutes", "min"),
    max_delay_minutes=("delay_minutes", "max"),
    weather_match_records=("temperature_2m", lambda values: values.notna().sum()),
)
daily_summary["avg_delay_minutes"] = daily_summary["avg_delay_minutes"].round(2)

route_group_columns = ["collection_date", "route_id", "route_short_name", "route_long_name", "service_type", "route_display_name", "route_corridor_name", "is_special_route"]
route_group_columns = [column for column in route_group_columns if column in decision_engine_output.columns]

route_daily_summary = decision_engine_output.groupby(route_group_columns, dropna=False, as_index=False).agg(
    records=("trip_id", "size"),
    unique_trips=("trip_id", "nunique"),
    avg_delay_minutes=("delay_minutes", "mean"),
    max_delay_minutes=("delay_minutes", "max"),
)
route_daily_summary["avg_delay_minutes"] = route_daily_summary["avg_delay_minutes"].round(2)

rank_source = decision_engine_output.copy()
if "is_special_route" in rank_source.columns:
    rank_source = rank_source[~rank_source["is_special_route"].fillna(False).astype(bool)].copy()

top_route_group_columns = ["route_id", "route_short_name", "route_long_name", "service_type", "route_display_name", "route_corridor_name", "is_special_route"]
top_route_group_columns = [column for column in top_route_group_columns if column in rank_source.columns]

top_delayed_routes = rank_source.groupby(top_route_group_columns, dropna=False, as_index=False).agg(
    records=("trip_id", "size"),
    avg_delay_minutes=("delay_minutes", "mean"),
    max_delay_minutes=("delay_minutes", "max"),
).sort_values(["avg_delay_minutes", "records"], ascending=[False, False]).head(20)
top_delayed_routes["avg_delay_minutes"] = top_delayed_routes["avg_delay_minutes"].round(2)

print("Daily summary shape:", daily_summary.shape)
print("Route daily summary shape:", route_daily_summary.shape)
print("Top delayed routes shape:", top_delayed_routes.shape)
if "is_special_route" in top_delayed_routes.columns:
    print("Special routes in default top delayed ranking:", int(top_delayed_routes["is_special_route"].fillna(False).astype(bool).sum()))
top_delayed_routes.head()


Daily summary shape: (31, 8)
Route daily summary shape: (12254, 12)
Top delayed routes shape: (20, 10)
Special routes in default top delayed ranking: 0


,route_id,route_short_name,route_long_name,service_type,route_display_name,route_corridor_name,is_special_route,records,avg_delay_minutes,max_delay_minutes
187,HUIA-404,HUIA,HUIA,Train / Rail,HUIA,HUIA - Hamilton / The Strand Auckland Via Puke...,False,146,35.14,119.950000
14,128-202,128,128,Bus,128,128 - Hibiscus Coast Station / Helensville,False,7012,8.44,40.566667
13,126-203,126,126,Bus,126,126 - Albany / Westgate Via Riverhead,False,7593,3.99,111.766667
128,807-202,807,807,Bus,807,807 - Devonport Wharf / Cheltenham Loop,False,7108,3.57,61.383333
78,376-203,376,376,Bus,376,376 - Papakura Shops / Auranga Dr Via Drury,False,10385,3.03,61.683333


In [37]:
# Intervention logic table held in memory only

intervention_logic = pd.DataFrame([
    {"delay_risk": "Low", "delay_minutes_rule": "0 to 5 minutes absolute delay", "recommended_action": "No operational action required", "operational_meaning": "Service is operating within an acceptable tolerance."},
    {"delay_risk": "Medium", "delay_minutes_rule": "More than 5 and up to 15 minutes absolute delay", "recommended_action": "Monitor route conditions", "operational_meaning": "Route should be watched for emerging disruption patterns."},
    {"delay_risk": "High", "delay_minutes_rule": "More than 15 and up to 25 minutes absolute delay", "recommended_action": "Adjust service headway", "operational_meaning": "Operations may need spacing or dispatch adjustments."},
    {"delay_risk": "Severe", "delay_minutes_rule": "More than 25 minutes absolute delay", "recommended_action": "Deploy standby bus or supervisor review", "operational_meaning": "High disruption risk that may need active intervention."},
])

intervention_logic


,delay_risk,delay_minutes_rule,recommended_action,operational_meaning
0,Low,0 to 5 minutes absolute delay,No operational action required,Service is operating within an acceptable tole...
1,Medium,More than 5 and up to 15 minutes absolute delay,Monitor route conditions,Route should be watched for emerging disruptio...
2,High,More than 15 and up to 25 minutes absolute delay,Adjust service headway,Operations may need spacing or dispatch adjust...
3,Severe,More than 25 minutes absolute delay,Deploy standby bus or supervisor review,High disruption risk that may need active inte...


### Sanity Check - Dashboard Tables

Expected result: the dashboard tables are generated in memory and use real Auckland route IDs. Dashboard files should be regenerated only after the validation checks above have passed and the changed outputs are reviewed in Git.


## Step 10 - Reproducible Export Configuration

The output paths below define the dashboard-ready files used by the later Streamlit and decision-support layers. Keep `WRITE_OUTPUTS = False` while inspecting or testing the notebook. Set it to `True` only when intentionally regenerating project outputs from the validated real Auckland pipeline.


In [38]:
# Optional dashboard-ready exports
# Keep False for inspection/test runs. Set True only for a controlled output refresh.
# Outputs are separated by dataset profile so all-file and model-baseline results cannot overwrite each other.
# When enabled, files are written to temporary CSVs first and then replaced after successful writes.

WRITE_OUTPUTS = True

OUTPUT_PROFILE = {
    "model_baseline_parquet": "model_baseline",
    "all_file_parquet": "all_file",
    "daily_csv_fallback": "daily_csv_fallback",
}.get(input_source_used, INPUT_DATASET)

OUTPUT_DIR = DATA_PROCESSED / "outputs" / OUTPUT_PROFILE
SUMMARY_OUTPUT_DIR = OUTPUT_DIR / "summaries"

output_paths = {
    "realtime_with_routes_sample": OUTPUT_DIR / "realtime_with_routes_sample.csv",
    "gtfs_weather_integrated_sample": OUTPUT_DIR / "gtfs_weather_integrated_sample.csv",
    "decision_engine_output": OUTPUT_DIR / "decision_engine_output.csv",
    "intervention_logic": OUTPUT_DIR / "intervention_logic.csv",
    "daily_summary": SUMMARY_OUTPUT_DIR / "gtfs_realtime_daily_summary.csv",
    "route_daily_summary": SUMMARY_OUTPUT_DIR / "gtfs_realtime_route_daily_summary.csv",
    "top_delayed_routes": SUMMARY_OUTPUT_DIR / "gtfs_realtime_top_delayed_routes.csv",
}

print("Output profile:", OUTPUT_PROFILE)
print("Output folder:", OUTPUT_DIR)

for name, path in output_paths.items():
    print(f"{name}: {path} | exists={path.exists()}")

def write_csv_safely(frame, path, chunksize=None):
    path = Path(path)
    temporary_path = path.with_name(path.name + ".tmp")

    try:
        if temporary_path.exists():
            temporary_path.unlink()

        frame.to_csv(temporary_path, index=False, chunksize=chunksize)
        temporary_path.replace(path)
        print(f"Wrote {path} | rows={len(frame):,}")
    except Exception:
        if temporary_path.exists():
            temporary_path.unlink()
        raise

if WRITE_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    SUMMARY_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    write_csv_safely(gtfs_routes.head(10000), output_paths["realtime_with_routes_sample"])
    write_csv_safely(gtfs_weather.head(10000), output_paths["gtfs_weather_integrated_sample"])
    write_csv_safely(decision_engine_output, output_paths["decision_engine_output"], chunksize=100000)
    write_csv_safely(intervention_logic, output_paths["intervention_logic"])
    write_csv_safely(daily_summary, output_paths["daily_summary"])
    write_csv_safely(route_daily_summary, output_paths["route_daily_summary"])
    write_csv_safely(top_delayed_routes, output_paths["top_delayed_routes"])

    print("Exports completed.")
else:
    print("WRITE_OUTPUTS is False. No files were written or overwritten.")


Output profile: all_file
Output folder: D:\Yoobee\Term 3\Capstone\ai-dss-auckland-public-transport\data\processed\outputs\all_file
realtime_with_routes_sample: D:\Yoobee\Term 3\Capstone\ai-dss-auckland-public-transport\data\processed\outputs\all_file\realtime_with_routes_sample.csv | exists=True
gtfs_weather_integrated_sample: D:\Yoobee\Term 3\Capstone\ai-dss-auckland-public-transport\data\processed\outputs\all_file\gtfs_weather_integrated_sample.csv | exists=True
decision_engine_output: D:\Yoobee\Term 3\Capstone\ai-dss-auckland-public-transport\data\processed\outputs\all_file\decision_engine_output.csv | exists=True
intervention_logic: D:\Yoobee\Term 3\Capstone\ai-dss-auckland-public-transport\data\processed\outputs\all_file\intervention_logic.csv | exists=True
daily_summary: D:\Yoobee\Term 3\Capstone\ai-dss-auckland-public-transport\data\processed\outputs\all_file\summaries\gtfs_realtime_daily_summary.csv | exists=True
route_daily_summary: D:\Yoobee\Term 3\Capstone\ai-dss-auckland-pu

## Final Reproducibility Checklist

Before committing regenerated outputs, confirm the following:

- The selected GTFS-Realtime input is the real Auckland repaired daily feed or a clearly documented fallback.
- Delay fields are numeric and unreasonable outliers have been filtered.
- Route metadata match rate is acceptable.
- Weather match rate is acceptable after UTC hourly alignment.
- Decision recommendations are generated from real Auckland GTFS records, not Kaggle prototype rows.
- Exported CSV files are reviewed with `git status` so the committed outputs match this real Auckland validation run.
